### 1. Create Tables (orders, order_items, products, product_category_name)

In [0]:
DROP TABLE ORDERS;
CREATE TABLE orders(
    order_id VARCHAR(32),
    customer_id VARCHAR(32),
    order_status VARCHAR(32),
    order_purchase_timestamp TIMESTAMP,
    order_approved_at TIMESTAMP,
    order_delivered_carrier_date TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP
);

In [0]:
DROP TABLE order_items;
CREATE TABLE order_items(
    order_id VARCHAR(32),
    order_item_id INTEGER,
    product_id VARCHAR(32),
    seller_id VARCHAR(32),
    shipping_limit_date TIMESTAMP,
    price DECIMAL,
    freight_value DECIMAL
);
SELECT COUNT(*) FROM order_items;

In [0]:
drop table products;
CREATE TABLE products(
    product_id VARCHAR(32),
    product_category_name VARCHAR(100),
    product_name_length INT,
    product_description_length INT,
    product_photos_qty INT,
    product_weight_g INT,
    product_length_cm INT,
    product_height_cm INT,
    product_width_cm INT
);

In [0]:
DROP TABLE product_category_name;
CREATE TABLE product_category_name(
    product_category_name VARCHAR(100),
    product_category_name_english VARCHAR(100)
);

In [0]:
DROP TABLE sellers;
CREATE TABLE sellers(
    seller_id VARCHAR(32),
    seller_zip_code_prefix INT,
    seller_city VARCHAR(64),
    seller_state VARCHAR(2)
);

### 2. Copy over data from S3

In [0]:
COPY orders
FROM 's3://ecommerce-supply-chain-381492252693/raw/olist_orders_dataset.csv'
IAM_ROLE default
CSV
IGNOREHEADER 1;

COPY order_items
FROM 's3://ecommerce-supply-chain-381492252693/raw/olist_order_items_dataset.csv'
IAM_ROLE default
CSV
IGNOREHEADER 1;

COPY sellers
FROM 's3://ecommerce-supply-chain-381492252693/raw/olist_sellers_dataset.csv'
IAM_ROLE default
CSV
IGNOREHEADER 1;

COPY products
FROM 's3://ecommerce-supply-chain-381492252693/raw/olist_products_dataset.csv'
IAM_ROLE default
CSV
IGNOREHEADER 1;

COPY product_category_name
FROM 's3://ecommerce-supply-chain-381492252693/raw/product_category_name_translation.csv'
IAM_ROLE default
CSV
IGNOREHEADER 1;

In [0]:
SELECT COUNT(*) FROM orders;
SELECT COUNT(*) FROM order_items;
SELECT COUNT(*) FROM sellers;
SELECT COUNT(*) FROM products;
SELECT COUNT(*) FROM product_category_name;
SELECT * FROM sys_load_error_detail ORDER BY start_time DESC LIMIT 10;

### 3. Fulfillment Rate by Month
- percent orders fulfilled on time by month/year
- Notice a large drop in apr 2017 and march 2018

In [0]:
SELECT 
    count(*) as total_delivered,
    DATE_TRUNC('month',order_purchase_timestamp) as purchased_month_year,
    SUM(CASE WHEN order_delivered_customer_date <= order_estimated_delivery_date
            THEN 1 ELSE 0 END)::FLOAT / count(*) *100 as percent_fulfilled_per_month
FROM orders

WHERE order_status = 'delivered'
AND order_delivered_customer_date is NOT NULL
GROUP BY purchased_month_year
HAVING total_delivered > 100
ORDER BY purchased_month_year;

### 4. Average Lead Time by Product Category
- Notice that office furniture has the highest operational cost with an average 20 day lead time.

In [0]:
SELECT 
    product_category_name.product_category_name_english as category_name,
    AVG(DATEDIFF('day', orders.order_purchase_timestamp, orders.order_delivered_customer_date)) as avg_lead_time
FROM orders
INNER JOIN order_items
ON orders.order_id = order_items.order_id
INNER JOIN products
ON order_items.product_id = products.product_id
INNER JOIN product_category_name
ON products.product_category_name = product_category_name.product_category_name
GROUP BY product_category_name.product_category_name_english
HAVING count(*) > 100
ORDER BY avg_lead_time DESC;

### 5. Order Volume Trends
- Count the distinct orders per month per category
- From a quick glance (without quicksight dashboard), you can see that furniture is consistently the highest ordered category per month

In [0]:
SELECT 
    COUNT(DISTINCT orders.order_id) as total_orders,
    DATE_TRUNC('month',orders.order_purchase_timestamp) as purchase_month_year,
    product_category_name.product_category_name_english as category
FROM orders
INNER JOIN order_items
ON orders.order_id = order_items.order_id
INNER JOIN products
ON order_items.product_id = products.product_id
INNER JOIN product_category_name
ON products.product_category_name=product_category_name.product_category_name
GROUP BY category,purchase_month_year
HAVING total_orders >=5 
ORDER BY purchase_month_year ASC, total_orders DESC

### 6. Order Status by Region
- Do different regions tend to have different statuses? (e.g. cancellations, missing, etc)

First, determine the distribution of different statuses

In [0]:
SELECT
    order_status,
    count(*) as counts
FROM orders
GROUP BY orders.order_status

Then compare statuses by region

In [0]:
SELECT
    sellers.seller_state as seller_state,
    orders.order_status as order_status,
    count(DISTINCT orders.order_id) as orders_count
FROM orders
INNER JOIN order_items
ON orders.order_id=order_items.order_id
INNER JOIN sellers
ON order_items.seller_id=sellers.seller_id
GROUP BY seller_state,order_status
ORDER BY seller_state ASC, order_status DESC